In [2]:
"""
SHAP plots in the selected-features space (post poly + SelectKBest).
Larger fonts and dots, includes dependence plots, grouped and full
beeswarm, and heatmap.

Requires: pip install shap
"""

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

# config
RESULTS_ROOT = "resultados_8modelos"
METHOD       = "random_search"
MODEL_NAME   = "XGBoost"

TOP_N_DEPENDENCE = 10     # how many dependence plots to generate (previously 5)
BEESWARM_MAX_DISPLAY = 12 # how many variables to show in the "grouped" beeswarm

# font and element sizes, larger so they're readable without zooming
FONT_TITLE  = 22
FONT_AXIS   = 18
FONT_TICK   = 16
FONT_LEGEND = 16
DOT_SIZE    = 90   # point size in dependence plots

plt.rcParams.update({
    "font.size":        FONT_AXIS,
    "axes.titlesize":   FONT_TITLE,
    "axes.labelsize":   FONT_AXIS,
    "xtick.labelsize":  FONT_TICK,
    "ytick.labelsize":  FONT_TICK,
    "legend.fontsize":  FONT_LEGEND,
    "figure.titlesize": FONT_TITLE,
})

SPLIT_PATH = os.path.join(RESULTS_ROOT, "particion_datos", "train_test_split.pkl")
MODEL_PATH = os.path.join(RESULTS_ROOT, METHOD, "models", f"{MODEL_NAME}.pkl")
SHAP_DIR   = os.path.join(RESULTS_ROOT, METHOD, "shap")
os.makedirs(SHAP_DIR, exist_ok=True)

# load split and pipeline
split = joblib.load(SPLIT_PATH)
X_train, X_test = split["X_train"], split["X_test"]

pipeline = joblib.load(MODEL_PATH)

poly     = pipeline.named_steps["poly"]
scaler   = pipeline.named_steps["scaler"]
selector = pipeline.named_steps["select"]
model    = pipeline.named_steps["model"]

# reconstruct the names of the selected features
poly_names      = poly.get_feature_names_out(X_train.columns)
selected_mask   = selector.get_support()
selected_names  = poly_names[selected_mask]
n_selected      = selected_mask.sum()

print(f"k = {selector.k}  ->  {n_selected} selected features")
print(list(selected_names))

# transform X_test up to just before the model
def transform_to_model_space(X):
    Xt = poly.transform(X)
    Xt = scaler.transform(Xt)
    Xt = selector.transform(Xt)
    return pd.DataFrame(Xt, columns=selected_names, index=X.index)

X_test_sel = transform_to_model_space(X_test)

# shap with treeexplainer
explainer = shap.TreeExplainer(model)
shap_expl = explainer(X_test_sel)

# plots
# grouped beeswarm (top variables + "sum of N other features")
plt.figure(figsize=(11, 0.55 * BEESWARM_MAX_DISPLAY + 2))
shap.plots.beeswarm(shap_expl, max_display=BEESWARM_MAX_DISPLAY, show=False)
plt.title(f"SHAP Beeswarm — {MODEL_NAME} (top {BEESWARM_MAX_DISPLAY})", fontsize=FONT_TITLE, fontweight="bold")
plt.xlabel("Impact on prediction (SHAP value)", fontsize=FONT_AXIS)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_beeswarm_top{BEESWARM_MAX_DISPLAY}.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: beeswarm_top{BEESWARM_MAX_DISPLAY}.png")

# full beeswarm (all selected variables, ungrouped)
plt.figure(figsize=(11, 0.55 * n_selected + 2))
shap.plots.beeswarm(shap_expl, max_display=n_selected, show=False)
plt.title(f"SHAP Beeswarm — {MODEL_NAME} (all {n_selected} variables)", fontsize=FONT_TITLE, fontweight="bold")
plt.xlabel("Impact on prediction (SHAP value)", fontsize=FONT_AXIS)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_beeswarm_completo.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("Saved: beeswarm_completo.png")

# bar plot (mean |SHAP| importance), all variables
plt.figure(figsize=(11, 0.5 * n_selected + 2))
shap.plots.bar(shap_expl, max_display=n_selected, show=False)
plt.title(f"Mean SHAP Importance — {MODEL_NAME}", fontsize=FONT_TITLE, fontweight="bold")
plt.xlabel("Mean |SHAP value|", fontsize=FONT_AXIS)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_bar.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: shap_bar.png")

# heatmap plot (instances x variables, ordered by clustering)
plt.figure(figsize=(12, 0.35 * n_selected + 3))
shap.plots.heatmap(shap_expl, show=False)
plt.title(f"SHAP Heatmap — {MODEL_NAME}", fontsize=FONT_TITLE, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: shap_heatmap.png")

# top-N dependence plots, larger dots
mean_abs_shap = np.abs(shap_expl.values).mean(axis=0)
top_n = min(TOP_N_DEPENDENCE, n_selected)
top_idx = np.argsort(mean_abs_shap)[::-1][:top_n]

for rank, i in enumerate(top_idx, start=1):
    feat = selected_names[i]
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.plots.scatter(shap_expl[:, feat], color=shap_expl, show=False, ax=ax)
    ax.set_title(f"SHAP Dependence — {feat}\n({MODEL_NAME}, rank #{rank})",
                 fontsize=FONT_TITLE - 2, fontweight="bold")
    ax.set_xlabel(feat, fontsize=FONT_AXIS)
    ax.set_ylabel("SHAP value", fontsize=FONT_AXIS)
    ax.tick_params(labelsize=FONT_TICK)
    # enlarge the scatter points (the first PathCollection is the point cloud)
    for coll in ax.collections:
        coll.set_sizes([DOT_SIZE])
    plt.tight_layout()
    safe_feat = feat.replace(" ", "_x_").replace("^", "_pow").replace("/", "_")
    fig.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_dependence_{rank:02d}_{safe_feat}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved {top_n} dependence plots: {[selected_names[i] for i in top_idx]}")

# waterfall for a single case
plt.figure(figsize=(10, 0.45 * n_selected + 2))
shap.plots.waterfall(shap_expl[0], show=False)
plt.title(f"SHAP Waterfall — {MODEL_NAME} (test example #0)", fontsize=FONT_TITLE, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_waterfall_ejemplo.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: waterfall_ejemplo.png")

print(f"\nAll plots saved to: {SHAP_DIR}")


k = 17  ->  17 selected features
['FC', 'FO', 'sF', 'FC^2', 'FC FO', 'FC sF', 'OC FO', 'FO InOr', 'FO F', 'FO sF', 'FO aO', 'FO C', 'FO sC', 'sF C', 'sF sC', 'sO C', 'sO sC']
Saved: beeswarm_top12.png
Saved: beeswarm_completo.png
Saved: shap_bar.png
Saved: shap_heatmap.png
Saved 10 dependence plots: ['sO C', 'sO sC', 'FO aO', 'FO C', 'FO', 'FO sF', 'FO InOr', 'FO sC', 'FC sF', 'FC']
Saved: waterfall_ejemplo.png

All plots saved to: resultados_8modelos\random_search\shap
